# Abacus xDESI Quick Pasting Validation

Single-GPU `NSIDE=1024` validation for the small `z<0.5`, `log10(M200c_hMsun)>14` Abacus halo catalog.

This notebook is the current recommended quick comparison workflow:

- repaste halo profiles to `5 * R200c`, using `abacus_pasting_config.yaml`;
- build the GODMAX theory with `gg_transition_model='poweradd'`, i.e. no response-suppressed `gg` comparison;
- build total-particle-shell `tau`, `kappa_cmb`, and `kappa_wl` fields from the Abacus HEALPix shell products;
- build direct non-halo field-particle maps from matched `total - halo` shell products;
- save and compare both the previous low-ell-replaced product and the new additive `pasted + direct field` product;
- compare all requested auto/cross spectra to theory.

Restart the kernel before running so the JAX GPU preallocation settings take effect.


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pathlib
import sys

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.95'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

NOTEBOOK_DIR = pathlib.Path.cwd()
if NOTEBOOK_DIR.name != 'xDESI':
    NOTEBOOK_DIR = pathlib.Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import gc

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import abacus_pasting_helpers as aph
import abacus_particle_shell_helpers as psh
from abacus_lightcone_catalog import preprocess_abacus_catalogs

CONFIG = NOTEBOOK_DIR / 'abacus_pasting_config.yaml'
CATALOG = 'zlt0p5_logMgt14p0'
NSIDE = 1024
PROFILE_PAINT_R200C_FACTOR = 5.0

TOTAL_ROOT = pathlib.Path('/mnt/ceph/users/backlight/AbacusBacklight_base_c9999_ph9999/lightcone_healpix/total')
HALO_ROOT = pathlib.Path('/mnt/storone/nfs1/backlight/AbacusBacklight_base_c9999_ph9999/lightcone_healpix/halo')
LOWELL_REPLACEMENT_ELL = 256
LOWELL_REPLACEMENT_FIELDS = ('map_kappa_cmb', 'map_kappa_wl', 'map_tau')
DIRECT_FIELD_FIELDS = ('map_kappa_cmb', 'map_kappa_wl', 'map_tau', 'map_ksz')
SHELL_WEIGHT_MODE = 'average'
SHELL_WEIGHT_NSAMPLES = 48
MAX_SHELLS = None
OVERWRITE_SHELL_CACHE = False

THEORY_GG_TRANSITION_MODEL = 'poweradd'  # recommended no-response gg comparison; use 'response' for the old setting
THEORY_LABEL = f'GODMAX theory (gg_transition_model={THEORY_GG_TRANSITION_MODEL})'

OVERWRITE_PASTED_MAPS = True
OVERWRITE_LOWELL_MAP = True
OVERWRITE_DIRECT_FIELD_CACHE = False
OVERWRITE_DIRECT_FIELD_MAP = True
ELL_MIN = 10
ELL_MAX = 3000
DELTA_ELL = 20

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})


## Check configuration and catalog

In [ ]:
cfg = aph.load_config(CONFIG)
paint_factor = float(cfg['pasting']['max_paint_R200c_factor'])
assert np.isclose(paint_factor, PROFILE_PAINT_R200C_FACTOR), (
    f'Config has max_paint_R200c_factor={paint_factor}, expected {PROFILE_PAINT_R200C_FACTOR}. '
    'Update abacus_pasting_config.yaml before repasting.'
)
assert cfg['godmax']['source_nz_bin_column'] == 'BIN5', cfg['godmax']['source_nz_bin_column']
print('config:', CONFIG)
print('source n(z) bin:', cfg['godmax']['source_nz_bin_column'])
print('paint radius:', paint_factor, '* R200c')
print('low-ell replacement:', LOWELL_REPLACEMENT_FIELDS, 'ell_cut=', LOWELL_REPLACEMENT_ELL)
print('direct field roots:', {'total': str(TOTAL_ROOT), 'halo': str(HALO_ROOT)})
print('direct field datasets:', DIRECT_FIELD_FIELDS)
print('theory:', THEORY_LABEL)

cat_path = aph.catalog_path(cfg, CATALOG)
if not cat_path.exists():
    preprocess_abacus_catalogs(CONFIG, only_catalogs=[CATALOG], overwrite=True)
print('catalog:', cat_path)
with __import__('h5py').File(cat_path, 'r') as f:
    cat_attrs = dict(f.attrs)
    print(cat_attrs)
    print('n_halos =', len(f['z']))


## Paste maps on one GPU

This overwrites the quick-validation one-split pasted map when `OVERWRITE_PASTED_MAPS=True`. The HDF5 attrs are checked so an old `3 * R200c` map cannot be used accidentally.

In [3]:
partial_path = aph.run_paste_split(
    CONFIG,
    CATALOG,
    split_index=0,
    num_splits=1,
    nside=NSIDE,
    overwrite=OVERWRITE_PASTED_MAPS,
)
final_pasted_path = aph.combine_partial_maps(
    CONFIG,
    CATALOG,
    num_splits=1,
    nside=NSIDE,
    overwrite=OVERWRITE_PASTED_MAPS,
)
maps_pasted, galaxies, pasted_attrs = aph.load_maps_h5(final_pasted_path)
assert np.isclose(float(pasted_attrs['max_paint_R200c_factor']), PROFILE_PAINT_R200C_FACTOR), pasted_attrs
print('pasted-only map:', final_pasted_path)
print('pasted attrs:', pasted_attrs)
print({k: (float(np.nanmin(v)), float(np.nanmax(v)), float(np.isfinite(v).mean())) for k, v in maps_pasted.items()})
print('galaxies', galaxies.shape, 'valid', int(np.count_nonzero(galaxies[:, 5] > 0.5)) if len(galaxies) else 0)

E0601 19:42:49.158864 1371104 ptx_compiler_helpers.cc:88] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.


[paste] catalog=zlt0p5_logMgt14p0 split=0/1 halos=140,661
[paste] chunk 1: halos 0:50,000


/mnt/home/spandey/miniconda3/envs/ili-sbi/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/mnt/home/spandey/miniconda3/envs/ili-sbi/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


[paste] pixel work pairs=17,790,605 time=8.2s


/mnt/home/spandey/miniconda3/envs/ili-sbi/lib/python3.10/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


[paste] chunk 2: halos 50,000:100,000
[paste] pixel work pairs=4,629,611 time=4.9s
[paste] chunk 3: halos 100,000:140,661
[paste] pixel work pairs=2,625,783 time=4.3s
[paste] wrote /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/outputs/abacus_maps/abacus_c9999_ph9999_xdesi/abacus_pasted_maps_zlt0p5_logMgt14p0_nside1024_split000of001.h5
[combine] wrote /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/outputs/abacus_maps/abacus_c9999_ph9999_xdesi/abacus_pasted_maps_zlt0p5_logMgt14p0_nside1024.h5
pasted-only map: /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/outputs/abacus_maps/abacus_c9999_ph9999_xdesi/abacus_pasted_maps_zlt0p5_logMgt14p0_nside1024.h5
pasted attrs: {'catalog_key': 'zlt0p5_logMgt14p0', 'catalog_path': '/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/outputs/abacus_halos/abacus_c9999_ph9999_zlt0p5_logMgt14p0_halos.h5', 'combined_from_num_splits': 1, 'max_paint_R200c_factor': 5.0, 'n_galaxies': 223411, 'n_input_halos': 140661, 'n_sp

## Build matching theory

In [4]:
nz_info, _, _ = aph.load_xdesi_fit_lens_info(cfg)
compare_bin = 0
z_edges = np.asarray(nz_info['z_edges_bins_lens'])
z_range = tuple(z_edges[compare_bin])
z_mean = 0.5 * (z_range[0] + z_range[1])
print('comparison bin:', compare_bin, 'z_range:', z_range)

cls_cmb = aph.build_theory_cls(
    CONFIG,
    CATALOG,
    is_cmb_lensing=True,
    log10_mass_min=14.0,
    z_max=0.5,
    gg_transition_model=THEORY_GG_TRANSITION_MODEL,
)
cls_wl = aph.build_theory_cls(
    CONFIG,
    CATALOG,
    is_cmb_lensing=False,
    log10_mass_min=14.0,
    z_max=0.5,
    gg_transition_model=THEORY_GG_TRANSITION_MODEL,
)
print('actual theory gg_transition_model:', getattr(cls_cmb, 'gg_transition_model', THEORY_GG_TRANSITION_MODEL))

ell_th = np.asarray(cls_cmb.ell_array)
tau_fac = aph.tau_theory_conversion(cls_cmb.cosmo_params, z_mean)
theory = {
    'gg': np.asarray(cls_cmb.Cl_gal_gal_tot_mat[:, compare_bin, compare_bin]),
    'gy': np.asarray(cls_cmb.Cl_gal_y_tot_mat[:, compare_bin]),
    'gtau': np.asarray(cls_cmb.Cl_gal_tau_tot_mat[:, compare_bin]) * tau_fac,
    'gkappa_cmb': np.asarray(cls_cmb.Cl_gal_kappa_tot_mat[:, compare_bin, 0]),
    'gkappa_wl': np.asarray(cls_wl.Cl_gal_kappa_tot_mat[:, compare_bin, 0]),
}
print({k: (float(np.nanmin(v)), float(np.nanmax(v))) for k, v in theory.items()})

comparison bin: 0 z_range: (0.2741959798994975, 0.4268643216080403)
actual theory gg_transition_model: poweradd
{'gg': (2.7719939393368577e-05, 0.0008779863599397049), 'gy': (3.330375475466173e-12, 1.4685554628350605e-10), 'gtau': (4.5230627407061524e-10, 8.572204127982253e-08), 'gkappa_cmb': (4.041676323462311e-08, 4.327023024665565e-06), 'gkappa_wl': (3.0337240892255685e-08, 3.26412467025004e-06)}


## Build total-shell and direct field-shell fields

The low-ell replacement map uses total-particle-shell modes only at low ell. This avoids additive double counting of selected halos while restoring large-scale modes missing from high-mass halo-only profile pasting.

The direct-field map is an additive product. It uses matched shell files from `total` and `halo`, constructs `counts_field = counts_total - counts_halo` and `momentum_field = counts_total * vlos_total - counts_halo * vlos_halo`, then adds only kappa/tau/kSZ field contributions to the pasted halo-profile maps. This is diagnostic for the high-mass quick catalog because the pasted halos are not the full identified-halo population removed by the `halo` shell.


In [ ]:
shell_meta_all = psh.discover_total_shells(TOTAL_ROOT, z_max_hint=0.5)
shell_meta = psh.select_shells(shell_meta_all, z_min=1.0e-4, z_max=0.5, max_shells=MAX_SHELLS)
if not shell_meta:
    raise RuntimeError('No particle shells selected for z<0.5')
print('selected total shells:', len(shell_meta), 'z coverage:', shell_meta[-1]['z_lo'], 'to', shell_meta[0]['z_hi'])
display(pd.DataFrame(shell_meta)[['step_id', 'z_lo', 'z_hi', 'chi_lo_hMpc', 'chi_hi_hMpc', 'nside_counts', 'nside_vel_los', 'mean_count_fine']])

cache_root = psh.particle_shell_cache_root(cfg, NSIDE)
cache_paths = {}
cache_rows = []
for i, meta in enumerate(shell_meta, 1):
    print(f'[{i}/{len(shell_meta)}] total shell cache {meta["step_id"]}')
    cache_path = psh.read_or_create_downgraded_shell_cache(
        meta,
        NSIDE,
        cache_root,
        overwrite=OVERWRITE_SHELL_CACHE,
    )
    cache_paths[meta['step_id']] = cache_path
    _, cache_attrs = psh.load_downgraded_shell_cache(cache_path, NSIDE)
    cache_rows.append({
        'step_id': meta['step_id'],
        'output_count_sum': cache_attrs['output_count_sum'],
        'rel_count_error': (cache_attrs['output_count_sum'] - cache_attrs['input_count_sum']) / max(cache_attrs['input_count_sum'], 1.0),
        'vel_nonzero_count_zero': cache_attrs['vel_nonzero_count_zero_fine_pixels'],
        'runtime_sec': cache_attrs.get('cache_runtime_sec', 0.0),
    })
display(pd.DataFrame(cache_rows))

weights = {
    meta['step_id']: psh.compute_shell_weights(
        meta,
        cls_cmb,
        cls_wl,
        mode=SHELL_WEIGHT_MODE,
        n_samples=SHELL_WEIGHT_NSAMPLES,
    )
    for meta in shell_meta
}
weight_rows = []
for meta in shell_meta:
    row = {'step_id': meta['step_id'], 'z_mid': meta['z_mid'], 'dchi_hMpc': meta['dchi_hMpc']}
    row.update({f'w_{k}': v for k, v in weights[meta['step_id']].items() if k != 'mode'})
    weight_rows.append(row)
display(pd.DataFrame(weight_rows))

total_shell_maps, shell_diag = psh.build_shell_field_maps(shell_meta, cache_paths, weights, NSIDE)
display(pd.DataFrame(shell_diag)[['step_id', 'mean_total_counts', 'mean_source_counts', 'source_mass_fraction', 'w_kappa_cmb', 'w_kappa_wl', 'w_tau']])
print('total shell maps:', {k: (float(np.nanmin(v)), float(np.nanmax(v)), float(np.isfinite(v).mean())) for k, v in total_shell_maps.items()})

matched_shell_meta = psh.discover_matched_total_halo_shells(TOTAL_ROOT, HALO_ROOT, z_min=1.0e-4, z_max=0.5, max_shells=MAX_SHELLS)
if [m['step_id'] for m in matched_shell_meta] != [m['step_id'] for m in shell_meta]:
    raise RuntimeError('Matched total-halo shells differ from selected total shell list.')
print('matched total-halo shells:', len(matched_shell_meta))

direct_cache_paths = {}
direct_cache_rows = []
for i, meta in enumerate(matched_shell_meta, 1):
    print(f'[{i}/{len(matched_shell_meta)}] direct field cache {meta["step_id"]}')
    direct_cache_path = psh.read_or_create_direct_field_shell_cache(
        meta,
        NSIDE,
        cache_root,
        overwrite=OVERWRITE_DIRECT_FIELD_CACHE,
    )
    direct_cache_paths[meta['step_id']] = direct_cache_path
    _, direct_attrs = psh.load_direct_field_shell_cache(direct_cache_path, NSIDE)
    direct_cache_rows.append({
        'step_id': meta['step_id'],
        'field_mass_fraction': direct_attrs['field_mass_fraction'],
        'halo_mass_fraction': direct_attrs['halo_mass_fraction'],
        'input_field_count_sum': direct_attrs['input_field_count_sum'],
        'output_field_count_sum': direct_attrs['output_field_count_sum'],
        'negative_fine': direct_attrs['negative_count_fine_pixels'],
        'negative_parent': direct_attrs['negative_count_parent_pixels'],
        'runtime_sec': direct_attrs.get('cache_runtime_sec', 0.0),
    })
display(pd.DataFrame(direct_cache_rows))

direct_weights = {
    meta['step_id']: psh.compute_dataset_shell_weights(
        meta,
        cls_cmb,
        cls_wl,
        wl_source_bins=(1,),
        mode=SHELL_WEIGHT_MODE,
        n_samples=SHELL_WEIGHT_NSAMPLES,
    )
    for meta in matched_shell_meta
}
direct_field_maps, direct_diag = psh.build_direct_field_maps(
    matched_shell_meta,
    direct_cache_paths,
    direct_weights,
    NSIDE,
    map_keys=DIRECT_FIELD_FIELDS,
)
direct_diag_df = pd.DataFrame(direct_diag)
display(direct_diag_df[['step_id', 'mean_total_counts', 'mean_source_counts', 'source_mass_fraction', 'field_mass_fraction_cache', 'negative_count_fine_pixels']])
print('direct field maps:', {k: (float(np.nanmin(v)), float(np.nanmax(v)), float(np.isfinite(v).mean())) for k, v in direct_field_maps.items()})


## Make and save latest comparison map products


In [ ]:
maps_latest = psh.harmonic_blend_maps(
    maps_pasted,
    total_shell_maps,
    NSIDE,
    LOWELL_REPLACEMENT_ELL,
    fields=LOWELL_REPLACEMENT_FIELDS,
)
maps_direct = psh.add_field_maps(maps_pasted, direct_field_maps)
for product_name, product_maps in [('lowell', maps_latest), ('direct_field', maps_direct)]:
    for key in product_maps:
        if not np.all(np.isfinite(product_maps[key])):
            raise ValueError(f'{product_name}:{key} contains non-finite values')

latest_map_path = (
    aph.map_run_dir(cfg)
    / f'abacus_pasted_maps_{CATALOG}_nside{NSIDE}_lowell_total_replace_L{LOWELL_REPLACEMENT_ELL}.h5'
)
latest_attrs = dict(pasted_attrs)
latest_attrs.update({
    'map_product': 'pasted_plus_total_shell_lowell_replacement',
    'base_pasted_map_path': str(final_pasted_path),
    'total_shell_root': str(TOTAL_ROOT),
    'total_shell_cache_root': str(cache_root),
    'lowell_replacement_ell_cut': int(LOWELL_REPLACEMENT_ELL),
    'lowell_replacement_fields_json': list(LOWELL_REPLACEMENT_FIELDS),
    'shell_weight_mode': SHELL_WEIGHT_MODE,
    'shell_weight_nsamples': int(SHELL_WEIGHT_NSAMPLES),
    'theory_gg_transition_model': THEORY_GG_TRANSITION_MODEL,
})
if latest_map_path.exists() and not OVERWRITE_LOWELL_MAP:
    maps_latest, galaxies_latest, latest_attrs = aph.load_maps_h5(latest_map_path)
else:
    aph.write_maps_h5(latest_map_path, maps_latest, galaxies, latest_attrs)
    galaxies_latest = galaxies

direct_map_path = (
    aph.map_run_dir(cfg)
    / f'abacus_pasted_maps_{CATALOG}_nside{NSIDE}_plus_direct_field_shells.h5'
)
direct_summary = {
    'n_shells': len(direct_diag),
    'field_mass_fraction_mean': float(np.nanmean([row['field_mass_fraction_cache'] for row in direct_diag])),
    'field_mass_fraction_min': float(np.nanmin([row['field_mass_fraction_cache'] for row in direct_diag])),
    'field_mass_fraction_max': float(np.nanmax([row['field_mass_fraction_cache'] for row in direct_diag])),
    'negative_count_fine_pixels_total': int(sum(row['negative_count_fine_pixels'] for row in direct_diag)),
}
direct_attrs = dict(pasted_attrs)
direct_attrs.update({
    'map_product': 'pasted_plus_direct_nonhalo_field_shells',
    'base_pasted_map_path': str(final_pasted_path),
    'total_shell_root': str(TOTAL_ROOT),
    'halo_shell_root': str(HALO_ROOT),
    'direct_field_cache_root': str(cache_root),
    'direct_field_fields_json': list(DIRECT_FIELD_FIELDS),
    'direct_field_normalization': 'delta_field=(counts_total-counts_halo-mean_field)/mean_total; kSZ=-Wtau*momentum_field/(mean_total*c)',
    'direct_field_velocity_interpretation': 'heal-vel-los is a mean velocity per occupied fine pixel; momentum is count-weighted before total-minus-halo subtraction',
    'direct_field_quick_catalog_caveat': 'Diagnostic for logM>14 quick catalog because halo shells remove all identified-halo particles, not only this high-mass pasted catalog.',
    'direct_field_summary_json': direct_summary,
    'shell_weight_mode': SHELL_WEIGHT_MODE,
    'shell_weight_nsamples': int(SHELL_WEIGHT_NSAMPLES),
    'theory_gg_transition_model': THEORY_GG_TRANSITION_MODEL,
})
if direct_map_path.exists() and not OVERWRITE_DIRECT_FIELD_MAP:
    maps_direct, galaxies_direct, direct_attrs = aph.load_maps_h5(direct_map_path)
else:
    aph.write_maps_h5(direct_map_path, maps_direct, galaxies, direct_attrs)
    galaxies_direct = galaxies
print('latest low-ell map product:', latest_map_path)
print('direct field map product:', direct_map_path)
print('direct field summary:', direct_summary)
print('lowell attrs:', latest_attrs)
print('direct attrs:', direct_attrs)
print('lowell maps:', {k: (float(np.nanmin(v)), float(np.nanmax(v)), float(np.isfinite(v).mean())) for k, v in maps_latest.items()})
print('direct maps:', {k: (float(np.nanmin(v)), float(np.nanmax(v)), float(np.isfinite(v).mean())) for k, v in maps_direct.items()})


## Visualize pasted, shell, and latest fields

In [ ]:
def mollview_percentile(field, title, *, symmetric=False, cmap='viridis', q=99.0):
    arr = np.asarray(field)
    finite = np.isfinite(arr)
    if not np.any(finite):
        raise ValueError(f'{title} has no finite pixels')
    if symmetric:
        vmax = float(np.nanpercentile(np.abs(arr[finite]), q))
        vmin = -vmax
    else:
        vmin = float(np.nanpercentile(arr[finite], 100.0 - q))
        vmax = float(np.nanpercentile(arr[finite], q))
    hp.mollview(arr, title=title, min=vmin, max=vmax, cmap=cmap)
    plt.show()

delta_g_all, mean_g, ngal = aph.make_galaxy_overdensity_map(galaxies_latest, NSIDE)
mollview_percentile(delta_g_all, 'delta_g', symmetric=True, cmap='coolwarm')
mollview_percentile(maps_latest['map_ymap'], 'y pasted', symmetric=False)
for key, label in [('map_tau', 'tau'), ('map_kappa_cmb', 'kappa_cmb'), ('map_kappa_wl', 'kappa_wl'), ('map_ksz', 'kSZ')]:
    mollview_percentile(maps_pasted[key], f'{label}: pasted only', symmetric=True, cmap='coolwarm')
    if key in total_shell_maps:
        mollview_percentile(total_shell_maps[key], f'{label}: total shell replacement field', symmetric=True, cmap='coolwarm')
    if key in direct_field_maps:
        mollview_percentile(direct_field_maps[key], f'{label}: direct non-halo field', symmetric=True, cmap='coolwarm')
    if key in maps_latest:
        mollview_percentile(maps_latest[key], f'{label}: low-ell L{LOWELL_REPLACEMENT_ELL}', symmetric=True, cmap='coolwarm')
    if key in maps_direct:
        mollview_percentile(maps_direct[key], f'{label}: pasted + direct field', symmetric=True, cmap='coolwarm')
print('all-bin galaxy mean per pixel:', mean_g, 'Ngal:', ngal)


## Measure spectra

In [ ]:
cls_by_variant = {
    'pasted_only': aph.measure_basic_cls(maps_pasted, galaxies_latest, NSIDE, z_range=z_range),
    f'latest_lowell_L{LOWELL_REPLACEMENT_ELL}': aph.measure_basic_cls(maps_latest, galaxies_latest, NSIDE, z_range=z_range),
    'pasted_plus_direct_field': aph.measure_basic_cls(maps_direct, galaxies_latest, NSIDE, z_range=z_range),
}
for name, cls in cls_by_variant.items():
    print(name, 'Ngal', cls['n_gal'], 'shot', cls['shot_gg'])


## Binned map-vs-theory comparisons

The comparison products are `latest_lowell_L256` and `pasted_plus_direct_field` for `gtau`, `gkappa_cmb`, `gkappa_wl`, and kSZ diagnostics. `gg` and `gy` are shown for context;    does not change the galaxy catalog or the pasted `y` map. kSZ is diagnostic only because this workflow has no current GODMAX kSZ theory curve.


In [ ]:
def theory_interp(field, ell):
    return np.interp(np.asarray(ell), ell_th, theory[field], left=np.nan, right=np.nan)

def paired_to_theory(cls, field):
    meas_key = 'gg_without_shot' if field == 'gg' else field
    ell = np.asarray(cls['ell'])
    return psh.bin_spectrum_pair(
        ell,
        cls[meas_key],
        theory_interp(field, ell),
        ell_min=ELL_MIN,
        ell_max=ELL_MAX,
        delta_ell=DELTA_ELL,
    )

def binned_single(ell, values):
    return psh.bin_spectrum(ell, values, ell_min=ELL_MIN, ell_max=ELL_MAX, delta_ell=DELTA_ELL)

plot_fields = [
    ('gg', 'gg without shot'),
    ('gy', 'g x y'),
    ('gtau', 'g x tau'),
    ('gkappa_cmb', 'g x kappa_cmb'),
    ('gkappa_wl', 'g x kappa_wl'),
]
variant_styles = {
    'pasted_only': {'color': 'tab:gray', 'ls': '--', 'lw': 1.0},
    f'latest_lowell_L{LOWELL_REPLACEMENT_ELL}': {'color': 'tab:blue', 'ls': '-', 'lw': 1.35},
    'pasted_plus_direct_field': {'color': 'tab:orange', 'ls': '-', 'lw': 1.25},
}

fig, axes = plt.subplots(2, len(plot_fields), figsize=(4.5 * len(plot_fields), 7.4), sharex='col')
for j, (field, label) in enumerate(plot_fields):
    ax = axes[0, j]
    rax = axes[1, j]
    theory_bin = None
    for name, cls in cls_by_variant.items():
        paired = paired_to_theory(cls, field)
        style = variant_styles[name]
        ax.plot(paired['ell'], paired['value_a'], marker='.', ms=2.7, label=name, **style)
        rax.plot(paired['ell'], paired['ratio'], marker='.', ms=2.7, label=name, **style)
        if theory_bin is None:
            theory_bin = {'ell': paired['ell'], 'value': paired['value_b']}
        if field == 'gg' and name == 'pasted_only':
            gg_with = psh.bin_spectrum_pair(
                cls['ell'],
                cls['gg_with_shot'],
                theory_interp('gg', cls['ell']),
                ell_min=ELL_MIN,
                ell_max=ELL_MAX,
                delta_ell=DELTA_ELL,
            )
            ax.plot(gg_with['ell'], gg_with['value_a'], color='0.65', lw=0.9, alpha=0.8, label='gg with shot')
            rax.plot(gg_with['ell'], gg_with['ratio'], color='0.65', lw=0.9, alpha=0.8)
    if theory_bin is not None:
        ax.plot(theory_bin['ell'], theory_bin['value'], color='k', lw=1.7, label=THEORY_LABEL)
    ax.set_title(label)
    ax.set_xlim(ELL_MIN, ELL_MAX)
    rax.set_xlim(ELL_MIN, ELL_MAX)
    ax.set_xscale('log')
    rax.set_xscale('log')
    if field == 'gg':
        ax.set_yscale('log')
    else:
        ax.set_yscale('symlog', linthresh=1.0e-12)
    rax.axhline(1.0, color='k', lw=0.8, alpha=0.65)
    rax.axhspan(0.8, 1.2, color='0.85', alpha=0.35)
    rax.set_ylim(0.0, 2.0)
    ax.grid(alpha=0.25)
    rax.grid(alpha=0.25)
    if j == 0:
        ax.set_ylabel(r'binned $C_\ell$')
        rax.set_ylabel('measured / theory')
    rax.set_xlabel(r'$\ell$')
    if j == 0:
        ax.legend(fontsize=8)
fig.suptitle(f'{CATALOG}, NSIDE={NSIDE}, paint={PROFILE_PAINT_R200C_FACTOR} R200c, low-ell replace L{LOWELL_REPLACEMENT_ELL}, Delta ell={DELTA_ELL}')
fig.tight_layout()
plt.show()


## kSZ diagnostic and ratio summary

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4.2))
for name, cls in cls_by_variant.items():
    ksz_bin = binned_single(cls['ell'], cls['gksz'])
    ax.plot(ksz_bin['ell'], ksz_bin['value'], marker='.', ms=3, lw=1.0, label=name)
ax.axhline(0.0, color='k', lw=0.8, alpha=0.6)
ax.set_xscale('log')
ax.set_yscale('symlog', linthresh=1.0e-12)
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell^{g\,kSZ}$')
ax.set_title('kSZ diagnostic only')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.show()

summary_rows = []
for name, cls in cls_by_variant.items():
    for field, _ in plot_fields:
        paired = paired_to_theory(cls, field)
        for lo, hi in [(80, 300), (300, 1000), (1000, 2500)]:
            mask = (paired['ell'] >= lo) & (paired['ell'] < hi) & np.isfinite(paired['ratio'])
            summary_rows.append({
                'variant': name,
                'field': field,
                'ell_range': f'{lo}-{hi}',
                'median_ratio': float(np.nanmedian(paired['ratio'][mask])) if np.any(mask) else np.nan,
                'mean_ratio': float(np.nanmean(paired['ratio'][mask])) if np.any(mask) else np.nan,
                'n_bins': int(np.count_nonzero(mask)),
            })
summary = pd.DataFrame(summary_rows)
display(summary)

print('Recommended comparison product:', latest_map_path)
print('Use latest_lowell_L256 for gtau, gkappa_cmb, and gkappa_wl; use pasted_only for y and kSZ diagnostics.')
